# 📡 Step 6 · Monitoring & Drift-Triggered Retraining

**Operations.** Watch the CLIP `embedding` feature for drift, then retrain the YOLOv8 detector automatically when production inputs shift.

`🗄️ image_embeddings_fv → 🔁 train_yolo (feature-store-driven) → 🏷️ facerecognition vN → 📝 inference logging → 📈 centroid drift → 🔁 retrain`

This notebook adds two layers on top of the tutorial:

1. **Feature monitoring** on the `embedding` feature (centroid drift + norm-distribution PSI) and on a scalar feature (`num_bboxes`).
2. **Model monitoring** on the **real `facerecognition` detector**: the same `train_yolo` job that trained it in notebook 2 re-runs it from the feature store, the deployed similarity service logs query embeddings under it, and drift in those embeddings triggers the job again.

> ℹ️ Run notebooks 1-5 and `predictor.ipynb` first, so `wider_face_files`, `image_embeddings`, the CLIP model, `facerecognition`, and the `similarimages` deployment all exist.

In [ ]:
import os

import hopsworks
import run_job

project = hopsworks.login()
fs = project.get_feature_store()
mr = project.get_model_registry()
job_api = project.get_job_api()
print(f"✅ Connected to project: {project.name}")

## 📈 Part A · Feature monitoring (drift visibility)

Feature-group monitoring resolves the detection and reference windows by commit-time travel, so no schema change is needed.

- **`embedding`** supports `centroid_distance` (L2 between window centroids) and distribution metrics over the per-row **norm** (PSI / KL / JS / Hellinger).
- **scalar features** use mean / PSI as usual.
- **distribution metrics over rolling windows** (`compare_on_distribution`) require `statistics_config.kll = True` on the feature group, so per-commit statistics runs persist the mergeable KLL sketch. Both groups enable it at creation (`create_fgs.py` for `wider_face_files`, notebook 4 for `image_embeddings`). For a group created before this change, retrofit once with `fg.statistics_config.kll = True; fg.update_statistics_config()`.

The feature to monitor is chosen on `.compare_on(...)` / `.compare_on_distribution(...)`, not on `create_feature_monitoring(...)`.

### 🧠 Embedding drift: centroid + norm PSI
Compare recent commits against a trailing 7-day baseline window.

In [ ]:
emb_fg = fs.get_feature_group("image_embeddings", version=1)

# Centroid drift on the embedding: recent commits vs a trailing baseline window
emb_fg.create_feature_monitoring(
    name="embedding_centroid_drift",
).with_detection_window(time_offset="1d") \
 .with_reference_window(time_offset="30d", window_length="7d") \
 .compare_on(feature_name="embedding", metric="centroid_distance", threshold=0.1).save()

# Norm-distribution drift (PSI over the per-row embedding L2 norm)
emb_fg.create_feature_monitoring(
    name="embedding_norm_psi",
).with_detection_window(time_offset="1d") \
 .with_reference_window(time_offset="30d", window_length="7d") \
 .compare_on_distribution(feature_name="embedding", metric="PSI", threshold=0.2).save()

print("✅ Embedding monitoring enabled: centroid_distance (>0.1) + norm PSI (>0.2)")

### 🔢 Scalar drift: faces per image
PSI on `num_bboxes` in the `wider_face_files` group.

In [ ]:
# Scalar example: drift in the number of faces per image
files_fg = fs.get_feature_group("wider_face_files", version=1)

files_fg.create_feature_monitoring(
    name="num_bboxes_drift",
).with_detection_window(time_offset="1d") \
 .with_reference_window(time_offset="30d", window_length="7d") \
 .compare_on_distribution(feature_name="num_bboxes", metric="PSI", threshold=0.2).save()

print("✅ Scalar monitoring enabled: num_bboxes PSI (>0.2)")

## 🤖 Part B · Model monitoring + drift-triggered retraining

`create_model_monitoring(...)` watches a deployed model's **inference logs**: it reads the feature view's logging feature group, filters by model name/version, and requires the monitored model version to carry a `training_dataset_version` as the drift reference.

`🖼️ query → 🧠 embedding → 📝 inference log → 📈 centroid drift → 🔁 train_yolo`

The monitored (and retrained) model is the real `facerecognition` detector. The wiring:

- 🗂️ a **logging-enabled feature view** over `embedding`. Drift in the CLIP embeddings of incoming images is the proxy for the detector's input distribution shifting.
- 🔁 one run of `train_yolo` in **feature-store-driven mode** (`retrain`): `train.py` rebuilds the training data from the current `wider_face_files` rows, creates the baseline training dataset on the feature view (the embedding population right now), and registers a `facerecognition` version linked to both.
- 📝 the redeployed similarity service logs each query embedding under that `facerecognition` version.
- 📈 monitoring compares the logged embeddings' centroid against the baseline training-dataset centroid. After 3 consecutive shifts it re-runs the **same `train_yolo` job**, which registers a fresh, re-baselined version.

The *served* model (CLIP) and the *monitored/retrained* model (`facerecognition`) differ on purpose: `model_retraining_job` is any job, and the query embeddings are a shared drift signal.

### 🗂️ Logging-enabled feature view
A logging-enabled view over `embedding`. The baseline training dataset is created by the training job on its first feature-store-driven run (next cells).

In [ ]:
# Logging-enabled feature view over the embedding feature
image_embeddings_fv = fs.get_or_create_feature_view(
    name="image_embeddings_fv",
    version=1,
    query=emb_fg.select(["embedding"]),
    logging_enabled=True,
)
print(f"✅ Feature view 'image_embeddings_fv' v{image_embeddings_fv.version} ready")

### 🔁 Training job
The same `train_yolo` job as notebook 2 (`run_job.ensure_train_yolo_job` reuses or registers it). One job, three uses: the initial fine-tune, the feature-store-driven run below, and every drift-triggered retrain.

In [ ]:
train_yolo_job = run_job.ensure_train_yolo_job(project)
train_yolo_job

### 🏷️ Switch the detector to feature-store-driven training
Notebook 2 fine-tuned from a static snapshot. Running the job once in `retrain` mode moves it onto the feature store: `train.py` rebuilds the dataset from the current `wider_face_files` rows, creates the baseline training dataset on `image_embeddings_fv`, and registers a `facerecognition` version **linked to the feature view + baseline** — the version model monitoring will watch. Drift-triggered retrains are re-executions of exactly this run.

In [ ]:
execution = run_job.run_and_print_logs(train_yolo_job, args="retrain")
assert execution.success, "train_yolo failed — see the logs above"

# The version to monitor: the feature-store-driven run just registered it
faces_model = max(mr.get_models("facerecognition"), key=lambda m: m.version)
assert faces_model.training_dataset_version is not None, (
    "Latest facerecognition version has no training dataset link — re-run the "
    "feature-view cell above, then this cell."
)
print(f"✅ Monitoring target: facerecognition v{faces_model.version} "
      f"(training dataset v{faces_model.training_dataset_version})")

### 📝 Logging predictor
Re-write the `similarimages` predictor so it logs each query embedding through the feature view, tagged with the monitored `facerecognition` version. The *served* model stays CLIP — served and monitored/retrained models may differ.

In [ ]:
%%writefile predict_similar_images.py
import os
import io
import base64

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import hopsworks

# The query embeddings are logged under the monitored detector model, whose linked
# feature view + baseline training dataset drive the drift comparison (see notebook 6).
MONITORED_MODEL_NAME = "facerecognition"


def get_image_embedding(image, processor, model, device):
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)
    # L2-normalize
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    return image_features.squeeze().cpu().tolist()


class Predict(object):
    def __init__(self, async_logger, model):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.project = hopsworks.login()
        self.fs = self.project.get_feature_store()

        # Resolve the latest feature-store-linked version so logged inference rows are
        # tagged with the version the monitoring config filters on. Versions without a
        # training dataset link (the notebook-2 snapshot runs) are skipped.
        mr = self.project.get_model_registry()
        self.monitored_model_version = max(
            m.version
            for m in mr.get_models(MONITORED_MODEL_NAME)
            if m.training_dataset_version is not None
        )

        # Feature group (file_path + embedding) used for the similarity search
        self.fg = self.fs.get_feature_group("image_embeddings", version=1)

        # Logging-enabled feature view (embedding only) used for inference logging + monitoring
        self.fv = self.fs.get_feature_view("image_embeddings_fv", version=1)
        self.fv.init_serving(feature_logger=async_logger)

        model_path = os.environ["MODEL_FILES_PATH"]
        self.model = CLIPModel.from_pretrained(model_path).to(self.device).eval()
        self.processor = CLIPProcessor.from_pretrained(model_path)
        print("Initialization complete")

    def predict(self, inputs):
        try:
            b64_image = inputs[0][0]
            if not b64_image:
                return {"error": "Missing image (base64 string) in request"}

            image = Image.open(io.BytesIO(base64.b64decode(b64_image))).convert("RGB")
            embedding = get_image_embedding(image, self.processor, self.model, self.device)

            # Similarity search against the indexed embeddings
            results = self.fg.find_neighbors(embedding, k=3)
            returned_files, returned_images = [], []
            for result in results:
                path = result[1][0]
                returned_files.append(path)
                with open(path, "rb") as f:
                    returned_images.append(base64.b64encode(f.read()).decode("utf-8"))

            # Log the query embedding so model monitoring can detect drift in production inputs
            self.fv.log(
                untransformed_features=[[embedding]],
                model_name=MONITORED_MODEL_NAME,
                model_version=self.monitored_model_version,
            )

            return {"file_names": returned_files, "images": returned_images}
        except Exception as e:
            return {"error": str(e)}

### 🚀 Redeploy the similarity service
The served model stays the CLIP `openaiclip_vit_base_patch32`; only the predictor script changes (it now logs query embeddings).

In [ ]:
ms = project.get_model_serving()
model_mr = mr.get_model("openaiclip_vit_base_patch32", version=1)

dataset_api = project.get_dataset_api()
uploaded = dataset_api.upload(
    "predict_similar_images.py", model_mr.model_files_path, overwrite=True
)
script_path = os.path.join("/Projects", project.name, uploaded)

# Replace any existing deployment of this name with the logging-enabled one
existing = ms.get_deployment("similarimages")
if existing is not None:
    try:
        existing.stop(await_stopped=120)
    except Exception as e:
        print(f"(stop) {e}")
    existing.delete()

deployment = model_mr.deploy(name="similarimages", script_file=script_path)
deployment.start(await_running=300)
deployment.get_state().describe()

### 🚨 Wire up drift-triggered retraining
`model.create_model_monitoring(...)` resolves the model's feature view and version from provenance — no need to repeat them. `.with_reference_training_dataset()` defaults to the model's linked baseline training dataset. After 3 consecutive centroid shifts, `train_yolo` re-runs with `"retrain"`.

In [ ]:
# Monitor the logged query embeddings for centroid drift against the model's baseline
# training dataset; after 3 consecutive shifts, run train_yolo (passes "retrain" to train.py).
faces_model.create_model_monitoring(
    name="detector_retrain_on_embedding_drift",
    retrain_model_after_num_shifts=3,
    model_retraining_job=train_yolo_job,
    model_retraining_job_execution_args="retrain",
).with_detection_window(time_offset="1d") \
 .with_reference_training_dataset() \
 .compare_on(
     feature_name="embedding",
     metric="centroid_distance",
     threshold=0.1,
 ).save()

print(f"✅ Model monitoring enabled on facerecognition v{faces_model.version}: "
      "retrain train_yolo after 3 consecutive centroid shifts (>0.1)")

## 🧪 Simulate embedding drift

To exercise the trigger without waiting for real drift, log a batch of "drifted" query embeddings (shifted away from the baseline) under the monitored `facerecognition` version, then trigger the monitoring run manually with `run_once()` instead of waiting for the daily cron. Each run that detects a shift increments the consecutive-shift counter; after 3, `train_yolo` launches with `"retrain"` and registers a new, re-baselined model version.

In [ ]:
import numpy as np

# A real embedding to perturb away from
base = np.asarray(emb_fg.read().iloc[0]["embedding"], dtype=np.float64)
dim = base.shape[0]
rng = np.random.default_rng(42)

# Build drifted, L2-normalized query embeddings
drifted_rows = []
for _ in range(100):
    shifted = base + rng.normal(0.6, 0.2, size=dim)
    shifted = shifted / np.linalg.norm(shifted)
    drifted_rows.append([shifted.tolist()])

image_embeddings_fv.log(
    untransformed_features=drifted_rows,
    model_name="facerecognition",
    model_version=faces_model.version,
)
image_embeddings_fv.materialize_log(wait=True)
print("Logged drifted embeddings under facerecognition "
      f"v{faces_model.version}.")

# Trigger a monitoring run now instead of waiting for the cron schedule.
# Re-run this line (awaiting each execution) to accumulate consecutive shifts.
config = faces_model.get_monitoring_configs()[0]
config.run_once()

## 📝 Notes

- **Version pinning.** The monitoring config pins `model_version` at save time, and the deployment resolves the version it logs under at startup. A drift-triggered retrain registers a **new** `facerecognition` version that is *not* automatically re-monitored — the existing config keeps evaluating the old version's logs. To move monitoring to the new version: redeploy the predictor (it picks up the latest linked version) and recreate the monitoring config against it.
- **Re-running the monitoring cell.** `save()` raises on a duplicate config name; delete the old config first via `faces_model.get_monitoring_configs()` or the UI.
- **Retraining needs a GPU node** available when the trigger fires — the `train_yolo` job requests 1 GPU.
- Thresholds (`centroid_distance` 0.1, PSI 0.2) and `retrain_model_after_num_shifts` (3) are starting points. CLIP embeddings are L2-normalized, so calibrate the centroid threshold against an observed no-drift baseline.
- This exercises the FSTORE-2048 embedding profiler + `centroid_distance` / norm-PSI path end to end, plus the FSTORE-2053 retraining trigger.